# [LAB] A2A로 친구 Agent들과 축구 약속 잡기

메시·호날두·음바페의 일정 Agent 세 명을 A2A 서버로 띄우고, 이 노트북의 내 Agent가 A2A로 세 친구에게 빈 시간을 물어 다 같이 축구할 시간을 잡습니다. 친구들의 일정은 재미로 지어낸 수업용 일정입니다(실제 선수 일정이 아닙니다). 보기 중 하나를 골라 주석을 풀어 코드를 완성하고, 완성된 코드는 실행하며 값을 바꿔 봅니다.

- **2장 친구 일정 Agent 세 명을 A2A 서버로 띄우기**: 서버 코드를 나눠 보며 보기를 고르고, 터미널에서 미리 들어 있는 완성 서버 파일을 띄웁니다. 서버 하나에 친구 세 명이 주소만 다르게 들어 있습니다.
- **3장 내 Agent가 친구들과 약속 잡기**: 친구들의 Agent Card를 읽고, 메시에게 먼저 직접 물어본 뒤, 세 친구에게 묻고 약속까지 잡는 내 Agent를 완성해 실행합니다.

셀은 위에서부터 차례로 실행합니다. 서버는 터미널에서 켜 둔 채로 3장을 진행합니다. 약속 잡기 한 번에 모델을 열 번 넘게 부르므로 결과가 나오기까지 1분 안팎이 걸립니다.

**목차**

- 1. 실습 방법
- 2. 친구 일정 Agent 세 명을 A2A 서버로 띄우기
  - 2.1 불러오기
  - 2.2 친구들의 이번 주 일정
  - 2.3 친구 Agent 만들기
  - 2.4 A2A AgentExecutor
  - 2.5 Agent Card
  - 2.6 Starlette 앱 조립
- 3. 내 Agent가 친구들과 약속 잡기
  - 3.1 친구 Agent Card 확인
  - 3.2 call_friend 함수
  - 3.3 메시에게 먼저 물어보기
  - 3.4 내 Agent(호스트) 만들기
  - 3.5 그래프 연결과 컴파일
  - 3.6 축구 약속 잡기
- 4. 정리

In [ ]:
from config.check import check_env

# 값은 출력하지 않고, 필요한 설정 이름이 있는지만 봅니다
check_env()

**볼 지점**
- "준비 완료"가 나오면 다음으로 넘어갑니다.
- "준비가 덜 되었습니다"가 나오면 ⛔ 줄의 → 안내대로 `.env`를 고칩니다. `.env`는 노트북을 시작할 때 한 번만 읽으므로, 고친 뒤에는 커널을 다시 시작하고 맨 위 셀부터 다시 실행합니다. 무엇을 고칠지 모르겠으면 그 셀의 출력을 AI에게 붙여 넣고 물어봅니다.
- 키 값은 화면에 나오지 않습니다. 혹시 나오면 셀 출력을 지우고, 그 내용을 다른 곳에 붙여 넣지 않습니다.

LangChain 디버그 모드를 켭니다. 체인과 그래프가 실행될 때 단계마다 입력과 출력이 셀 아래에 출력됩니다.

도구를 부를 때 `Error in ConsoleCallbackHandler.on_tool_start callback: KeyError('input')` 같은 줄이 섞여 보일 수 있습니다. 디버그 내용을 출력하는 쪽의 문제라 실행 결과에는 영향이 없으니 그대로 두면 됩니다.

In [ ]:
from langchain_core.globals import set_debug

# 출력이 너무 길면 True를 False로 바꾸고 이 셀을 다시 실행합니다
set_debug(True)

## 1. 실습 방법

코드 셀에서 손댈 곳은 주석으로 표시되어 있습니다.

- `# ✍️ CHANGE_HERE:` 바로 아래 줄의 값을 바꿔 다시 실행해 봅니다.
- `# ✅ CHOOSE_HERE:` 아래에 보기가 주석으로 들어 있습니다. 줄 끝의 번호(1️⃣~4️⃣)와 설명을 보고 하나를 고릅니다. 같은 번호가 붙은 줄을 모두 선택하고 `Ctrl + /`를 눌러 주석을 푼 뒤 실행합니다.

실행한 뒤에는 **성공 기준**과 비교해 봅니다. 🟢는 되어야 하는 것, ⛔는 다시 해야 하는 신호입니다. ⛔이면 푼 줄을 다시 선택하고 `Ctrl + /`로 주석 처리한 뒤 다른 보기를 고릅니다. 마지막으로 접힌 "정답 보기"에서 보기마다의 해설을 읽습니다.

**막혔을 때 AI에게 도움받는 프롬프트 힌트** 오류가 난 셀의 코드와 오류 메시지 전체를 붙여 넣고, 원인과 고칠 곳을 쉬운 말로 알려 달라고 합니다.

## 2. 친구 일정 Agent 세 명을 A2A 서버로 띄우기

친구마다 일정을 확인하는 도구(`check_schedule`)와 약속을 잡는 도구(`book_match`)를 가진 작은 LangGraph Agent를 만들고, 다른 Agent가 요청할 수 있도록 A2A 서버로 감쌉니다. 서버 하나에 친구 세 명을 주소만 다르게 답니다(`/messi/`, `/ronaldo/`, `/mbappe/`). 서버는 노트북 셀에서 돌리지 않고 터미널에서 띄웁니다.

이 폴더에는 정답이 모두 들어간 서버 파일 `a2a_server.py`가 미리 들어 있습니다. 아래 셀은 그 파일의 코드를 나눠 보여 주는 곳입니다. 셀에서 보기를 골라 실행하면 노트북 안에서만 실행되고, 서버 파일은 바뀌지 않습니다. 그래서 고른 보기가 틀려도 터미널의 서버는 그대로 동작합니다.

셀은 위에서부터 차례로 실행합니다. 오류 없이 틀리는 보기도 있으니, 고른 뒤에는 정답 보기의 해설로 확인합니다.

**왜 배우나:** 안쪽 Agent는 지금까지 만든 LangGraph Agent와 같습니다. 여기에 AgentExecutor, Agent Card, RequestHandler 세 가지를 더하면 다른 Agent가 부를 수 있는 서버가 됩니다. 친구 세 명은 같은 코드에 일정 데이터만 다릅니다.

### 2.1 불러오기

In [ ]:
"""
A2A 서버: 친구 일정 Agent 세 명(메시·호날두·음바페)을 A2A로 띄우기

A2A(Agent-to-Agent)는 Google이 제안한 Agent 사이 통신 표준이다.
서버 하나에 친구 Agent 세 명을 주소만 다르게 띄운다.
  http://127.0.0.1:8001/messi/   http://127.0.0.1:8001/ronaldo/   http://127.0.0.1:8001/mbappe/
일정은 수업용으로 지어낸 것이다(실제 선수 일정이 아니다).
"""

import uuid
from typing import Annotated

import a2a.types as t
import uvicorn
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events import EventQueue
from a2a.server.request_handlers.default_request_handler_v2 import (
    DefaultRequestHandlerV2,
)
from a2a.server.routes.agent_card_routes import create_agent_card_routes
from a2a.server.routes.jsonrpc_routes import create_jsonrpc_routes
from a2a.server.tasks import InMemoryTaskStore
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from starlette.applications import Starlette
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv();

**성공 기준**

- 🟢 셀이 오류 없이 실행된다(불러오기만 해서 출력은 없다)

### 2.2 친구들의 이번 주 일정

**✍️ CHANGE_HERE: 호날두 Agent의 말투를 바꿔 봅니다**

처음에는 그대로 두고 3장 끝까지 진행합니다. 그다음 친구 Agent 쪽을 바꿔 봅니다. 서버는 이 셀이 아니라 폴더의 `a2a_server.py` 파일로 돌기 때문에, VS Code 왼쪽 탐색기에서 노트북과 같은 폴더의 `a2a_server.py`를 열고 호날두(`"ronaldo"`) 아래 `"style":` 줄의 따옴표 안을 `아주 짧게, 한 문장으로만 말하는 말투`로 바꿔 저장합니다(`Ctrl + S`). 서버 터미널에서 **Ctrl + C**로 끈 뒤 다시 띄우고, 3장의 축구 약속 잡기 셀을 다시 실행합니다. 내 Agent의 코드는 그대로인데 호날두의 답만 달라지는지 봅니다. 확인했으면 원래대로 되돌려 저장하고 서버를 다시 띄웁니다.

In [ ]:
# ── 친구들의 이번 주 일정 (수업용으로 지어낸 일정) ──────────────────────────────
# 요일마다 세 시간대(10시·14시·19시)가 있다. busy에 적히지 않은 시간은 비어 있다.
DAYS = ["월", "화", "수", "목", "금", "토", "일"]
TIMES = ["10시", "14시", "19시"]

FRIENDS = {
    "messi": {
        "name": "메시",
        "style": "차분하고 겸손한 말투",
        "busy": {
            "월": {"10시": "팀 훈련", "19시": "인터 마이애미 경기"},
            "화": {"10시": "팀 훈련", "14시": "광고 촬영", "19시": "자선 행사"},
            "수": {"10시": "팀 훈련", "19시": "대표팀 영상 미팅"},
            "목": {"10시": "팀 훈련", "14시": "재활 운동"},
            "금": {"10시": "팀 훈련", "19시": "가족 저녁"},
            "토": {"14시": "가족 나들이", "19시": "리그 경기"},
            "일": {"10시": "가족 브런치"},
        },
    },
    "ronaldo": {
        "name": "호날두",
        # ✍️ CHANGE_HERE: 호날두 Agent의 말투를 바꿔 봅니다
        "style": "자신감 넘치는 말투로, 마지막에 'Siuuu!'를 붙인다",
        "busy": {
            "월": {"10시": "헬스장", "14시": "광고 촬영", "19시": "야간 헬스장"},
            "화": {"10시": "헬스장", "19시": "알나스르 경기"},
            "수": {"10시": "헬스장"},
            "목": {"10시": "헬스장", "14시": "인터뷰"},
            "금": {"10시": "헬스장", "14시": "수영", "19시": "다큐 촬영"},
            "토": {"14시": "사인회", "19시": "팬 미팅"},
            "일": {"14시": "회복 훈련", "19시": "알나스르 경기"},
        },
    },
    "mbappe": {
        "name": "음바페",
        "style": "빠르고 밝은 말투",
        "busy": {
            "월": {"10시": "팀 훈련", "19시": "챔피언스리그 경기"},
            "화": {"10시": "팀 훈련"},
            "수": {"10시": "팀 훈련", "14시": "스폰서 행사", "19시": "게임 방송"},
            "목": {"10시": "팀 훈련"},
            "금": {"10시": "팀 훈련", "19시": "대표팀 소집"},
            "토": {"19시": "라리가 경기"},
            "일": {"10시": "회복 훈련", "14시": "가족 식사", "19시": "영화 보기"},
        },
    },
}

**성공 기준**

- 🟢 셀이 오류 없이 실행된다(일정 데이터만 만들어 출력은 없다)

### 2.3 친구 Agent 만들기

**✅ 친구의 두 도구를 LLM과 ToolNode에 연결해 주세요.**

- 🟢 셀이 오류 없이 실행되고, 고른 보기에서 모델과 `ToolNode`가 같은 도구 목록 `tools`를 받는다
- ⛔ 셀에서 `ValueError`가 나거나 두 줄이 받는 도구가 서로 다르다 → 다른 보기를 고릅니다

In [ ]:
# ── 친구 Agent 만들기 ─────────────────────────────────────────────────────────
# 친구마다 같은 모양의 작은 LangGraph Agent를 만든다. 도구는 그 친구의 일정만 본다.


class State(TypedDict):
    messages: Annotated[list, add_messages]


def make_friend_agent(friend: str):
    info = FRIENDS[friend]
    booked = {}  # 서버가 켜져 있는 동안 잡은 약속({요일: {시간: 약속}}). 서버를 다시 띄우면 비워진다

    @tool
    def check_schedule() -> str:
        """이번 주(월~일) 10시·14시·19시 일정을 알려 준다. 일정이 없으면 '비어 있음'."""
        print(f"[{friend}] check_schedule()")
        week = []
        for day in DAYS:
            plans = {**info["busy"].get(day, {}), **booked.get(day, {})}
            week.append(
                f"{day}: "
                + ", ".join(f"{time} {plans.get(time, '비어 있음')}" for time in TIMES)
            )
        return "\n".join(week)

    @tool
    def book_match(day: str, time: str, host: str) -> str:
        """비어 있는 시간에 축구 약속을 잡는다. day는 '목'처럼 요일 한 글자, time은 10시·14시·19시 중 하나, host는 약속을 잡는 사람."""
        print(f"[{friend}] book_match({day}, {time}, {host})")
        day, time = day.strip()[:1], time.strip()
        if day not in DAYS or time not in TIMES:
            return (
                "요일은 월~일 한 글자, 시간은 10시·14시·19시 중 하나로 다시 요청합니다."
            )
        plan = booked.get(day, {}).get(time) or info["busy"].get(day, {}).get(time)
        if plan:
            return f"{day} {time}에는 이미 '{plan}' 일정이 있어 잡을 수 없습니다."
        booked.setdefault(day, {})[time] = f"축구 약속({host})"
        return f"{day} {time}에 {host}와(과) 축구 약속을 잡았습니다."

    tools = [check_schedule, book_match]
    # ✅ CHOOSE_HERE: 친구의 두 도구를 LLM과 ToolNode에 연결해 주세요.
    # (같은 번호의 줄을 모두 선택하고 Ctrl + /)
    # llm = get_llm().bind_tools(tools)       # 1️⃣ 도구 연결 + 도구 하나로 ToolNode
    # tool_node = ToolNode(check_schedule)    # 1️⃣
    #
    # llm = get_llm()                         # 2️⃣ 모델만 준비 + ToolNode
    # tool_node = ToolNode(tools)             # 2️⃣
    #
    # llm = get_llm().bind_tools(tools)       # 3️⃣ 도구 연결 + 같은 도구 목록으로 ToolNode
    # tool_node = ToolNode(tools)             # 3️⃣
    #
    # llm = get_llm().bind_tools(tools)       # 4️⃣ 도구 연결 + 일정 확인 도구만 ToolNode에
    # tool_node = ToolNode([check_schedule])  # 4️⃣

    system = SystemMessage(
        content=(
            f"너는 {info['name']}의 일정 비서 Agent다. 일정은 수업용으로 지어낸 것이다. "
            "일정을 물으면 check_schedule로 확인하고, 물어본 시간대에 비어 있는 요일을 빠짐없이 말한다. "
            "약속을 잡아 달라고 하면 바로 book_match로 잡는다(book_match가 비었는지 스스로 확인한다). "
            f"답은 한국어 두세 문장으로, {info['style']}으로 한다."
        )
    )

    def agent_node(state: State) -> dict:
        return {"messages": [llm.invoke([system] + state["messages"])]}

    builder = StateGraph(State)
    builder.add_node("agent", agent_node)
    builder.add_node("tools", tool_node)
    builder.add_edge(START, "agent")
    builder.add_conditional_edges("agent", tools_condition)
    builder.add_edge("tools", "agent")
    return builder.compile()


friend_agents = {friend: make_friend_agent(friend) for friend in FRIENDS}

<details><summary>정답 보기</summary>

친구 Agent가 쓸 모델과 도구 실행 노드를 준비합니다. 모델에는 두 도구(`check_schedule`, `book_match`)를 알려 주어 도구 호출을 요청할 수 있게 하고, 그 요청을 실제로 실행할 `ToolNode`도 같은 도구 목록으로 만듭니다. 이 셀 맨 아래 줄이 친구 세 명의 Agent를 이 함수로 만들기 때문에, 틀린 보기는 이 셀에서 바로 오류가 나기도 합니다.

**정답: 3️⃣**

- 1️⃣ `ToolNode`에는 도구 목록을 넘겨야 합니다. 목록 대신 도구 하나를 넘기면 이 셀 맨 아래에서 친구 Agent를 만들 때 `ValueError: The first argument must be a string or a callable with a __name__ for tool decorator. Got <class 'tuple'>`가 납니다.
- 2️⃣ 모델이 두 도구를 몰라 일정을 확인하지 않습니다. 셀도 서버도 오류 없이 돌지만, 서버 터미널에 `check_schedule()`·`book_match(…)` 줄이 한 줄도 나오지 않습니다. 사외 실행에서는 친구들의 답이 비어 있었는데도 내 Agent가 근거 없이 '약속이 잡혔다'고 답했습니다. 조용히 틀리는 경우라 서버 터미널을 꼭 봅니다.
- 3️⃣ `bind_tools`로 모델에 두 도구를 알려 주고, 같은 도구 목록으로 도구를 실제로 실행할 `ToolNode`를 만듭니다. 이 코드로 만든 서버라면 3장에서 물을 때 서버 터미널에 `[messi] check_schedule()`, 약속을 잡을 때 `[messi] book_match(목, 19시, 나)` 같은 줄이 나옵니다.
- 4️⃣ 모델은 두 도구를 알지만 실제로 실행하는 `ToolNode`에는 `book_match`가 없습니다. 일정 확인은 되지만 약속을 잡으려 하면 도구가 실행되지 않아, 서버 터미널에 `book_match(…)` 줄이 나오지 않습니다. 사외 실행에서는 그런데도 친구 Agent 하나가 '바로 잡았어!'라고 답했습니다. 모델의 말보다 서버 터미널의 도구 호출 줄을 믿습니다.

</details>

### 2.4 A2A AgentExecutor

**✅ 친구 Agent를 비동기로 실행하고 마지막 답을 꺼내 주세요.**

- 🟢 셀이 오류 없이 실행되고, 고른 보기가 `await`로 `self.agent`를 실행해 마지막 메시지의 글을 `answer`에 담는다
- ⛔ 정답 보기의 번호와 다르다 → 셀은 오류 없이 실행되어도 서버에서는 틀리니, 해설을 읽고 다른 보기를 고릅니다

In [ ]:
# ── A2A AgentExecutor ─────────────────────────────────────────────────────────
# AgentExecutor.execute(): A2A 요청(RequestContext)에서 메시지 글을 읽어
# 친구 Agent를 실행하고, 답을 A2A Message로 감싸 EventQueue에 보낸다.


class FriendExecutor(AgentExecutor):
    def __init__(self, agent):
        self.agent = agent  # 이 실행기가 맡은 친구 Agent

    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        # context.get_user_input(): 받은 메시지의 텍스트 파트를 모아 문자열로 돌려준다
        user_text = context.get_user_input() or ""

        # ✅ CHOOSE_HERE: 친구 Agent를 비동기로 실행하고 마지막 답을 꺼내 주세요.
        # (같은 번호의 줄을 모두 선택하고 Ctrl + /)
        # result = self.agent.ainvoke(                        # 1️⃣ await 없이 ainvoke
        #     {"messages": [HumanMessage(content=user_text)]}  # 1️⃣
        # )                                                   # 1️⃣
        # answer = result["messages"][-1].content             # 1️⃣
        #
        # result = await self.agent.ainvoke(                  # 2️⃣ 첫 메시지의 내용을 꺼냄
        #     {"messages": [HumanMessage(content=user_text)]}  # 2️⃣
        # )                                                   # 2️⃣
        # answer = result["messages"][0].content              # 2️⃣
        #
        # result = await self.agent.ainvoke(                  # 3️⃣ 마지막 메시지를 통째로 꺼냄
        #     {"messages": [HumanMessage(content=user_text)]}  # 3️⃣
        # )                                                   # 3️⃣
        # answer = result["messages"][-1]                     # 3️⃣
        #
        # result = await self.agent.ainvoke(                  # 4️⃣ await로 실행 + 마지막 메시지의 내용
        #     {"messages": [HumanMessage(content=user_text)]}  # 4️⃣
        # )                                                   # 4️⃣
        # answer = result["messages"][-1].content             # 4️⃣

        # t.Message: A2A 응답 메시지. t.Part(text=...): 텍스트 파트
        await event_queue.enqueue_event(
            t.Message(
                role=t.Role.ROLE_AGENT,
                parts=[t.Part(text=answer)],
                task_id=context.task_id,
                context_id=context.context_id,
                message_id=str(uuid.uuid4()),
            )
        )

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        pass

<details><summary>정답 보기</summary>

AgentExecutor는 A2A 요청을 실제 Agent 실행으로 잇습니다. 친구마다 실행기를 하나씩 만들고, 실행기는 맡은 친구 Agent를 `self.agent`에 들고 있습니다. 받은 글(`user_text`)을 사람 메시지로 감싸 `self.agent`를 **비동기로** 실행하고, 결과 메시지 목록의 마지막 내용을 `answer`에 담습니다. 바로 아래 코드가 `answer`를 A2A Message로 감싸 돌려보냅니다.

**정답: 4️⃣**

- 1️⃣ `await`가 없으면 실행하지 않고 실행 예약(coroutine)만 받습니다. 셀은 오류 없이 실행되지만, 이 코드로 만든 서버에 물으면 서버 터미널에 `TypeError: 'coroutine' object is not subscriptable`이 나오고, 3.3에서 `InternalError: 'coroutine' object is not subscriptable`이 납니다.
- 2️⃣ 첫 메시지는 내가 보낸 질문입니다. 셀은 오류 없이 실행되지만, 이 코드로 만든 서버라면 친구 Agent가 일정을 다 확인하고도(서버 터미널에 `check_schedule()` 줄이 나옴) 3.3에 `메시: 이번 주 저녁 19시에 비는 날 알려 줘.`처럼 보낸 질문이 그대로 돌아옵니다.
- 3️⃣ 글이 아니라 메시지 객체라서 `t.Part(text=answer)`에 넣을 수 없습니다. 셀은 오류 없이 실행되지만, 이 코드로 만든 서버에 물으면 서버 터미널에 `TypeError: bad argument type for built-in operation`이 나오고, 3.3에서 `InternalError`가 납니다.
- 4️⃣ 받은 글을 사람 메시지로 감싸 `await`로 비동기 실행하고, 마지막 메시지(친구 Agent의 최종 답)의 글을 `answer`에 담습니다.

</details>

### 2.5 Agent Card

In [ ]:
# ── Agent Card ────────────────────────────────────────────────────────────────
# Agent Card: 이 Agent가 누구이고 무엇을 할 수 있는지 적은 자기소개(메타데이터).
# 다른 Agent는 <주소>/.well-known/agent-card.json에서 이 카드를 읽어 친구를 찾는다(discover).


def make_card(friend: str) -> t.AgentCard:
    info = FRIENDS[friend]
    return t.AgentCard(
        name=f"{info['name']} 일정 Agent",
        description=f"{info['name']}의 이번 주 일정을 알려 주고, 비어 있는 시간에 축구 약속을 잡습니다(수업용 가상 일정).",
        version="1.0.0",
        capabilities=t.AgentCapabilities(streaming=False),
        default_input_modes=["text"],
        default_output_modes=["text"],
        supported_interfaces=[t.AgentInterface(url=f"http://127.0.0.1:8001/{friend}/")],
        skills=[
            t.AgentSkill(
                id="soccer_schedule",
                name="일정 확인·축구 약속",
                description="이번 주 빈 시간을 알려 주고, 비어 있는 시간에 축구 약속을 잡습니다.",
                tags=["schedule", "soccer"],
                examples=[
                    "이번 주 저녁 19시에 비는 날 알려 줘",
                    "목요일 19시에 축구 약속 잡아 줘",
                ],
            )
        ],
    )

**성공 기준**

- 🟢 셀이 오류 없이 실행된다(카드를 만드는 함수만 정의해 출력은 없다)

### 2.6 Starlette 앱 조립

In [ ]:
# ── Starlette 앱 조립 ─────────────────────────────────────────────────────────
# 서버 하나에 친구 세 명을 주소만 다르게 단다: /messi/, /ronaldo/, /mbappe/
# 친구마다 카드 주소(/<친구>/.well-known/agent-card.json)와 요청을 받는 주소(/<친구>/)가 있다.

routes = []
for friend, agent in friend_agents.items():
    card = make_card(friend)
    handler = DefaultRequestHandlerV2(
        agent_executor=FriendExecutor(agent),
        task_store=InMemoryTaskStore(),
        agent_card=card,
    )
    routes += create_agent_card_routes(
        card, card_url=f"/{friend}/.well-known/agent-card.json"
    )
    routes += create_jsonrpc_routes(handler, rpc_url=f"/{friend}/")

app = Starlette(routes=routes)

아래는 서버를 터미널에서 띄울 때만 실행되는 부분입니다. 노트북에서는 실행하지 않고 읽어만 봅니다.

```python
if __name__ == "__main__":
    print("친구 일정 A2A 서버 시작: http://localhost:8001")
    for friend in FRIENDS:
        print(
            f"  {FRIENDS[friend]['name']}: http://localhost:8001/{friend}/.well-known/agent-card.json"
        )
    # 127.0.0.1: 이 PC에서 오는 요청만 받는다. 0.0.0.0이면 다른 PC의 요청도 받는다.
    uvicorn.run(app, host="127.0.0.1", port=8001)
```

**성공 기준**

- 🟢 셀이 오류 없이 실행된다(앱만 만들고 서버는 띄우지 않는다)

이 폴더에는 정답이 모두 들어간 서버 파일(`a2a_server.py`)이 미리 들어 있습니다. 2.1부터 위 셀들은 그 파일의 코드를 나눠 보여 주는 곳입니다. 셀을 실행해 오류가 없는지 확인합니다(보기가 있는 셀은 먼저 고릅니다). 셀이 파일을 고치지는 않습니다.

서버는 터미널에서 이 완성 파일로 띄웁니다.

1. VS Code 메뉴 **터미널 → 새 터미널**을 엽니다(저장소 맨 위 폴더에서 열립니다).
2. 아래 명령을 붙여 넣고 실행합니다.

```
uv run python course/Chapter02/notebooks/a2a_server.py
```

3. 서버 시작 문구가 나오고 명령이 끝나지 않고 기다리면 켜진 것입니다. 이 터미널은 닫지 말고 노트북으로 돌아옵니다. 서버를 끌 때는 터미널에서 **Ctrl + C**를 누릅니다.

**볼 지점**
- 서버를 띄운 터미널에 먼저 아래 네 줄이 나옵니다. 서버 파일 맨 끝의 `print`가 출력한 것입니다. 그 아래에 서버 프로그램(uvicorn)이 `INFO:`로 시작하는 줄을 이어서 출력하고, 마지막 줄에 `Uvicorn running on`과 `127.0.0.1:8001`이 보입니다(`127.0.0.1`은 이 PC에서 오는 요청만 기다린다는 뜻입니다).

```
친구 일정 A2A 서버 시작: http://localhost:8001
  메시: http://localhost:8001/messi/.well-known/agent-card.json
  호날두: http://localhost:8001/ronaldo/.well-known/agent-card.json
  음바페: http://localhost:8001/mbappe/.well-known/agent-card.json
```

- 서버 하나에 친구 세 명이 주소만 다르게 들어 있습니다. 브라우저 주소창에 위 카드 주소 하나를 넣으면 그 친구의 Agent Card(JSON)가 보입니다.
- 3장에서 요청을 보내면 이 터미널에 `"POST /messi/ HTTP/1.1" 200 OK`처럼 어느 친구에게 요청이 왔는지 접속 기록이 나옵니다. 친구 Agent가 도구를 쓰면 `[messi] check_schedule()`, `[messi] book_match(목, 19시, 나)` 같은 줄도 나옵니다.
- 잡은 약속은 서버가 켜져 있는 동안만 기억합니다. 서버를 Ctrl+C로 끄고 다시 띄우면 처음 일정으로 돌아갑니다.
- 클라이언트에서 오류가 나면 **이 터미널부터** 봅니다. 서버 코드에서 난 오류(Traceback)가 여기에 나옵니다.

**성공 기준**

- 🟢 시작 문구 네 줄이 보이고, 마지막 줄에 `Uvicorn running on`과 `127.0.0.1:8001`이 보이며, 명령이 끝나지 않고 기다린다
- ⛔ `SyntaxError` 같은 오류가 나오고 바로 끝난다 → `a2a_server.py`의 일정을 고쳤다면 따옴표·쉼표·중괄호의 짝이 맞는지 보고 저장한 뒤 다시 띄웁니다
- ⛔ `address already in use` 또는 `10048`(Windows)이 나온다 → 8001 포트를 쓰는 서버가 이미 켜져 있습니다. 앞서 띄운 서버 터미널에서 Ctrl+C로 끄고 다시 띄웁니다

## 3. 내 Agent가 친구들과 약속 잡기

이제 이 노트북이 내 Agent(호스트)가 됩니다. 터미널에 띄운 친구 서버에 A2A로 접속해 친구들의 Agent Card를 읽고, 메시지를 보내고, 마지막에는 세 친구에게 묻고 약속까지 잡는 Agent를 만듭니다.

**이 장을 시작하기 전에 2장의 서버가 터미널에서 켜져 있어야 합니다.**

- **3.1 친구 Agent Card 확인:** 모델을 부르지 않고 카드(자기소개)만 읽습니다.
- **3.2~3.3:** 친구 한 명에게 메시지를 보내는 함수를 완성하고, 메시에게 먼저 물어봅니다.
- **3.4~3.6:** 그 함수를 도구(`ask_friend`)로 쓰는 내 Agent를 만들어, 세 친구가 모두 비는 시간에 약속을 잡습니다.

**왜 배우나:** 내 Agent는 친구들의 일정이 어디에 어떻게 저장되어 있는지 모릅니다. 카드와 메시지 형식만 맞추면 다른 Agent에게 묻고 일을 맡길 수 있다는 것이 A2A의 핵심입니다. 친구 Agent들을 서로 다른 팀이 만든 Agent라고 생각하면 됩니다.

준비 코드입니다. A2A 클라이언트와 LangGraph 도구를 불러오고, 친구 서버 주소(`SERVER_URL`)와 친구 목록을 정합니다. 접혀 있으니 펼치지 않아도 되고, 실행만 합니다.

In [ ]:
"""
A2A 클라이언트: 내 Agent가 친구 Agent 세 명에게 일정을 물어 축구 약속 잡기 (노트북 셀용)
"""

import uuid
from typing import Annotated, Literal

import a2a.types as t
import httpx
from a2a.client.base_client import BaseClient
from a2a.client.card_resolver import A2ACardResolver
from a2a.client.client import ClientConfig
from a2a.client.transports.jsonrpc import JsonRpcTransport
from a2a.utils.constants import PROTOCOL_VERSION_1_0, VERSION_HEADER
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv()

SERVER_URL = "http://127.0.0.1:8001"
FRIENDS = ["messi", "ronaldo", "mbappe"]
# A2A-Version 헤더: 서버가 1.x 형식 요청임을 알아보게 한다(없으면 예전 0.3 형식으로 보고 거절한다)
HEADERS = {VERSION_HEADER: PROTOCOL_VERSION_1_0}

### 3.1 친구 Agent Card 확인

In [ ]:
# ── 친구 Agent Card 확인 ──────────────────────────────────────────────────────
# 친구마다 <주소>/.well-known/agent-card.json을 읽는다. 모델은 부르지 않는다
async with httpx.AsyncClient(headers=HEADERS) as http:
    for friend in FRIENDS:
        card = await A2ACardResolver(http, f"{SERVER_URL}/{friend}").get_agent_card()
        print(f"[{friend}] {card.name}: {card.description}")
        print(f"    스킬: {[skill.name for skill in card.skills]}")

**성공 기준**

- 🟢 `[messi] 메시 일정 Agent: …`, `[ronaldo] 호날두 일정 Agent: …`, `[mbappe] 음바페 일정 Agent: …` 세 줄과 줄마다 `스킬: ['일정 확인·축구 약속']`이 출력되었다
- ⛔ `ConnectError`(연결 거부)가 난다 → 서버 터미널이 켜져 있는지 봅니다

### 3.2 call_friend 함수

**✅ t.SendMessageRequest를 생성해주세요.**

- 🟢 3.3에서 `메시:` 뒤에 저녁 19시에 비는 요일을 알려 주는 답이 나온다
- ⛔ 3.3에서 `InvalidParamsError`나 `TypeError`가 나거나 답이 질문과 어긋난다 → 다른 보기를 고르고 이 셀부터 다시 실행합니다

In [ ]:
# ── 친구 한 명에게 A2A로 메시지 보내기 ─────────────────────────────────────────
async def call_friend(friend: str, query: str) -> str:
    """A2A 프로토콜로 친구 Agent에게 메시지를 보내고 답 글을 돌려준다."""
    url = f"{SERVER_URL}/{friend}"
    async with httpx.AsyncClient(
        headers=HEADERS, timeout=120.0
    ) as http:  # 친구 Agent도 모델을 불러 답하므로 넉넉히 기다린다
        # 1. Agent Card 조회
        card = await A2ACardResolver(http, url).get_agent_card()

        # 2. JSON-RPC 트랜스포트로 클라이언트 생성
        transport = JsonRpcTransport(http, card, f"{url}/")
        client = BaseClient(card, ClientConfig(), transport, [])

        # 3. 메시지 전송
        # ✅ CHOOSE_HERE: t.SendMessageRequest를 생성해주세요.
        # (같은 번호의 줄을 모두 선택하고 Ctrl + /)
        # req = t.SendMessageRequest(                # 1️⃣ 사용자 메시지 + 텍스트 파트 + 고유 id
        #     message=t.Message(                     # 1️⃣
        #         role=t.Role.ROLE_USER,             # 1️⃣
        #         parts=[t.Part(text=query)],        # 1️⃣
        #         message_id=str(uuid.uuid4()),      # 1️⃣
        #     )                                      # 1️⃣
        # )                                          # 1️⃣
        #
        # req = t.SendMessageRequest(                # 2️⃣ 사용자 메시지 + 텍스트 파트
        #     message=t.Message(                     # 2️⃣
        #         role=t.Role.ROLE_USER,             # 2️⃣
        #         parts=[t.Part(text=query)],        # 2️⃣
        #     )                                      # 2️⃣
        # )                                          # 2️⃣
        #
        # req = t.SendMessageRequest(                # 3️⃣ 텍스트 파트에 "query" 글자
        #     message=t.Message(                     # 3️⃣
        #         role=t.Role.ROLE_USER,             # 3️⃣
        #         parts=[t.Part(text="query")],      # 3️⃣
        #         message_id=str(uuid.uuid4()),      # 3️⃣
        #     )                                      # 3️⃣
        # )                                          # 3️⃣
        #
        # req = t.SendMessageRequest(message=query)  # 4️⃣ query 글을 바로 message에 넣음

        async for response in client.send_message(req):
            # 응답의 message: 친구 Agent가 돌려준 Message
            if response.HasField("message"):
                return "".join(
                    part.text for part in response.message.parts if part.text
                )

    return "응답을 받지 못했습니다."

<details><summary>정답 보기</summary>

`call_friend`는 Agent Card 조회 → Client 생성 → 메시지 전송 → 응답에서 텍스트 꺼내기를 한 함수에 담았습니다. 고를 곳은 세 번째 단계입니다. 받은 `query`를 사용자 역할의 `t.Message`에 텍스트 파트로 담고, 메시지마다 고유한 id를 붙여 `SendMessageRequest`로 감싼 `req`를 만듭니다. 2장의 "A2A AgentExecutor" 셀에서 서버가 답을 보낼 때 쓴 `t.Message(...)`와 비교해 봅니다. 이 셀은 함수만 만들고 아무것도 출력하지 않습니다. 실제 호출은 아래 "메시에게 먼저 물어보기" 셀에서 합니다.

**정답: 1️⃣**

- 1️⃣ 받은 `query`를 사용자 역할의 메시지에 텍스트 파트로 담고, 고유한 `message_id`를 붙여 요청으로 감쌉니다.
- 2️⃣ `message_id`가 없어 서버가 요청을 받지 않습니다. 3.3에서 `InvalidParamsError: Validation failed`가 나고, 서버 터미널에는 Traceback도 `check_schedule()` 줄도 없습니다.
- 3️⃣ 따옴표 때문에 변수 값이 아니라 `query`라는 글자를 보냅니다. 오류는 없지만 친구 Agent는 무엇을 묻는지 몰라, 19시만이 아니라 한 주의 빈 시간 전체를 늘어놓는 식으로 질문과 어긋난 답을 합니다.
- 4️⃣ `message`에는 `t.Message`를 넣어야 합니다. 3.3에서 `TypeError: Parameter to initialize message field must be dict or instance of same class: expected <class 'a2a_pb2.Message'> got <class 'str'>.`가 납니다. 서버까지 가지 않아 서버 터미널에는 아무것도 나오지 않습니다.

</details>

### 3.3 메시에게 먼저 물어보기

In [ ]:
# ── 메시에게 먼저 물어보기 ─────────────────────────────────────────────────────
messi_answer = await call_friend("messi", "이번 주 저녁 19시에 비는 날 알려 줘.")
print(f"메시: {messi_answer}")

**성공 기준**

- 🟢 `메시:` 뒤에 목요일과 일요일 19시가 비어 있다는 답이 나온다(말투는 실행할 때마다 조금씩 다르다)
- 🟢 서버 터미널에 `[messi] check_schedule()` 줄과 `"POST /messi/ HTTP/1.1" 200 OK` 줄이 새로 나왔다
- ⛔ `InvalidParamsError`나 `TypeError`가 난다 → 3.2에서 고른 보기를 다른 보기로 바꾸고 3.2 셀부터 다시 실행합니다(서버는 그대로 둡니다)

### 3.4 내 Agent(호스트) 만들기

In [ ]:
# ── 내 Agent(호스트) 만들기 ───────────────────────────────────────────────────
# 내 Agent는 친구의 일정을 모른다. ask_friend 도구로 친구 Agent에게 A2A로 물어볼 뿐이다.
@tool
async def ask_friend(
    friend: Literal["messi", "ronaldo", "mbappe"], message: str
) -> str:
    """친구 일정 Agent에게 A2A로 메시지를 보내고 답을 받는다. friend는 messi, ronaldo, mbappe 중 하나."""
    print(f"[나 → {friend}] {message}")
    try:
        answer = await call_friend(friend, message)
    except Exception as e:  # 친구 Agent가 늦거나 꺼져 있으면 셀을 멈추지 않고, 답이 없다는 사실을 내 Agent에게 알린다
        answer = f"(응답 없음: {type(e).__name__}) {friend} Agent가 답하지 않았습니다."
    print(f"[{friend} → 나] {answer}")
    return answer


class State(TypedDict):
    messages: Annotated[list, add_messages]


host_llm = get_llm().bind_tools([ask_friend])
HOST_SYSTEM = SystemMessage(
    content=(
        "너는 내 축구 약속을 잡아 주는 비서 Agent다. 친구는 messi, ronaldo, mbappe 세 명이다. "
        "1) ask_friend로 세 친구 모두에게 한꺼번에 요청한 시간대에 비는 요일을 묻는다. "
        "2) 세 친구의 답에 모두 들어 있는 요일만 후보로 삼는다. 한 명이라도 안 되는 요일은 후보가 아니다. "
        "3) 후보가 있으면 하나를 골라, 세 친구 모두에게 한꺼번에 ask_friend로 그 요일·시간에 약속을 잡아 달라고 한다. "
        "약속을 잡는 사람(host)은 '나'라고 메시지에 적는다. "
        "4) 후보가 없거나 답하지 않은 친구가 있으면 약속을 잡지 말고, 친구별로 비는 요일(또는 답이 없었다는 것)을 알려 준다. "
        "5) 마지막에 정한 요일과 시간을 한 줄로 알려 준다."
    )
)


async def host_node(state: State) -> dict:
    return {"messages": [await host_llm.ainvoke([HOST_SYSTEM] + state["messages"])]}

**성공 기준**

- 🟢 셀이 오류 없이 실행된다(도구와 노드만 정의해 출력은 없다)

### 3.5 그래프 연결과 컴파일

**✅ tools_condition으로 도구 호출 여부를 분기하고 tools 이후 host로 돌아오게 연결해 주세요.**

- 🟢 셀이 오류 없이 실행되고, 고른 보기에 `tools`에서 `host`로 돌아오는 엣지와 `tools_condition` 갈림길이 함께 있다
- ⛔ 셀에서 `TypeError`나 `NameError`가 나거나, 3.6에서 친구에게 묻기만 하고 약속을 못 잡는다 → 다른 보기를 고릅니다

In [ ]:
builder = StateGraph(State)
builder.add_node("host", host_node)
builder.add_node("tools", ToolNode([ask_friend]))
builder.add_edge(START, "host")
# ✅ CHOOSE_HERE: tools_condition으로 도구 호출 여부를 분기하고 tools 이후 host로 돌아오게 연결해 주세요.
# (같은 번호의 줄을 모두 선택하고 Ctrl + /)
# builder.add_conditional_edges("host", tools_condition)    # 1️⃣ 도구 호출 여부로 분기만
#
# builder.add_conditional_edges("host", tools_condition)    # 2️⃣ 도구 호출 여부로 분기 + tools → host
# builder.add_edge("tools", "host")                         # 2️⃣
#
# builder.add_conditional_edges("host", tools_condition())  # 3️⃣ tools_condition을 불러 결과를 넘김
# builder.add_edge("tools", "host")                         # 3️⃣
#
# builder.add_edge("host", END)                             # 4️⃣ host → END + tools → host
# builder.add_edge("tools", "host")                         # 4️⃣
host_agent = builder.compile()

<details><summary>정답 보기</summary>

노드 두 개(`host`, `tools`)와 시작 엣지는 이미 있습니다. host 다음에는 도구 호출 요청이 있으면 tools로, 없으면 끝으로 가게 나누고, tools를 실행한 뒤에는 다시 host로 돌아오게 잇습니다. 이 고리가 있어야 내 Agent가 친구들의 답을 보고 다음 일(약속 잡기)을 이어서 합니다.

**정답: 2️⃣**

- 1️⃣ tools(친구에게 묻기)를 실행한 뒤 갈 곳이 없어 거기서 끝납니다. 셀은 오류 없이 실행되지만, 3.6의 맨 끝이 내 Agent의 답(`Ai Message`)이 아니라 친구의 답이 든 `Tool Message`이고, 약속은 잡지 못합니다. 서버 터미널에 `book_match(…)` 줄이 나오지 않습니다.
- 2️⃣ 도구 호출 요청이 있으면 tools로, 없으면 끝으로 갑니다. tools(친구에게 묻기)를 실행한 뒤에는 host로 돌아가 친구들의 답을 보고 다음 일(약속 잡기)을 정합니다. 이 되돌아가는 엣지가 순환의 고리입니다.
- 3️⃣ 괄호를 붙이면 State 없이 그 자리에서 함수를 부릅니다. 셀을 실행할 때 `TypeError: tools_condition() missing 1 required positional argument: 'state'`가 납니다.
- 4️⃣ host 다음은 언제나 끝입니다. 셀은 오류 없이 실행되지만, 3.6에서 내 Agent가 친구에게 물으려는 도구 호출(`Tool Calls: ask_friend …`)만 남기고 끝납니다. 친구에게 아무것도 보내지 않아 `[나 → messi]` 같은 줄도, 서버 터미널의 새 기록도 없습니다.

</details>

### 3.6 축구 약속 잡기

**✍️ CHANGE_HERE: 요청을 주말 오전으로 바꿔 봅니다**

먼저 그대로 실행해 결과를 봅니다. 그다음 `# ✍️ CHANGE_HERE:` 바로 아래 줄의 따옴표 안 `저녁 19시`를 `주말 오전 10시`로 바꿔 이 셀을 다시 실행합니다. 서버는 다시 띄우지 않아도 됩니다. 같은 Agent가 요청만 바뀌어 다른 시간을 찾는지 봅니다.

**✅ 내 Agent를 비동기로 실행해 주세요.**

- 🟢 셀이 1분 안팎 돌고, 맨 끝 `Ai Message`에 세 친구가 모두 비는 요일과 시간으로 약속을 잡았다는 답이 나온다
- ⛔ `TypeError`나 `InvalidUpdateError`가 나거나 결과가 출력되지 않는다 → 다른 보기를 고르고 이 셀을 다시 실행합니다

In [ ]:
# ── 축구 약속 잡기 ─────────────────────────────────────────────────────────────
# ✍️ CHANGE_HERE: 요청을 주말 오전으로 바꿔 봅니다
request = "이번 주 저녁 19시에 다 같이 축구 한 판 하자. 약속을 잡아 줘."
# ✅ CHOOSE_HERE: 내 Agent를 비동기로 실행해 주세요.
# (같은 번호의 줄을 모두 선택하고 Ctrl + /)
# host_agent_result = host_agent.invoke(                 # 1️⃣ await 없이 invoke
#     {"messages": [HumanMessage(content=request)]}      # 1️⃣
# )                                                      # 1️⃣
#
# host_agent_result = host_agent.ainvoke(                # 2️⃣ await 없이 ainvoke
#     {"messages": [HumanMessage(content=request)]}      # 2️⃣
# )                                                      # 2️⃣
#
# host_agent_result = await host_agent.ainvoke(          # 3️⃣ await로 ainvoke + 메시지 딕셔너리
#     {"messages": [HumanMessage(content=request)]}      # 3️⃣
# )                                                      # 3️⃣
#
# host_agent_result = await host_agent.ainvoke(request)  # 4️⃣ 요청 글을 그대로 넘김

results = host_agent_result["messages"]
for m in results:
    m.pretty_print()

**성공 기준**

- 🟢 1분 안팎(길면 2분) 돈 뒤, 맨 끝 `Ai Message`에 **목요일 19시**로 약속을 잡았다는 한 줄이 나온다
- 🟢 그 위에 `[나 → messi] …`, `[messi → 나] …`처럼 세 친구와 두 번씩(빈 시간 묻기, 약속 잡기) 주고받은 메시지가 보인다
- 🟢 서버 터미널에 친구마다 `check_schedule()` 줄과 `book_match(목, 19시, …)` 줄이 나온다
- 🟢 (요청을 주말 오전 10시로 바꾼 뒤) 맨 끝에 **토요일 10시**로 약속을 잡았다는 답이 나온다
- 🟢 (2.2에서 호날두 말투를 바꾸고 서버를 다시 띄운 뒤) `[ronaldo → 나]` 답만 한 문장으로 짧아지고 `Siuuu!`가 빠지며, 약속 시간은 그대로 목요일 19시다
- ⛔ `TypeError`나 `InvalidUpdateError`가 난다 → 이 셀에서 고른 보기를 다시 봅니다
- ⛔ 친구에게 묻기만 하고 끝나거나, 맨 끝에 `Tool Calls: ask_friend`만 남고 끝난다 → 3.5에서 고른 엣지 보기를 다시 봅니다
- ⛔ 내 Agent는 약속을 잡았다는데 서버 터미널에 `book_match(…)` 줄이 없다 → 내 Agent의 말보다 서버 기록을 믿습니다. 셀을 한 번 더 실행하고, 그래도 없으면 출력 전체를 AI에게 붙여 넣고 원인을 물어봅니다
- ⛔ `InternalError`가 나고 메시지에 `429`나 `Too Many Requests`가 보인다 → 모델 서버가 잠시 요청을 막은 것입니다. 1분쯤 기다렸다가 이 셀을 다시 실행합니다

<details><summary>정답 보기</summary>

내 Agent의 도구 `ask_friend`와 노드 `host_node`는 비동기(`async`) 함수입니다. 그래서 그래프도 `ainvoke`로 실행하고 `await`로 끝날 때까지 기다립니다. 입력은 요청 글을 사람 메시지로 감싼 `{"messages": [...]}` 딕셔너리입니다.

**정답: 3️⃣**

- 1️⃣ `ask_friend`와 `host_node`가 비동기 함수라 동기 실행(`invoke`)으로는 돌릴 수 없습니다. `TypeError: No synchronous function provided to "host".`가 나고, 오류 안내에 `invoke via the async API (ainvoke, astream, etc.)`라고 나옵니다.
- 2️⃣ `await`가 없으면 실행하지 않고 실행 예약(coroutine)만 받습니다. 바로 아래 줄에서 `TypeError: 'coroutine' object is not subscriptable`이 납니다. 친구에게는 아무것도 보내지 않습니다.
- 3️⃣ 비동기 그래프를 `ainvoke`로 실행하고 `await`로 끝날 때까지 기다립니다. 입력은 요청 글을 `HumanMessage`로 감싼 `{"messages": [...]}` 딕셔너리입니다.
- 4️⃣ 그래프의 입력은 State 모양의 딕셔너리(`{"messages": [...]}`)여야 합니다. 글을 그대로 넘기면 `InvalidUpdateError: Expected dict, got 이번 주 저녁 19시에 …`가 납니다.

</details>

**볼 지점**
- "친구 Agent Card 확인" 셀은 모델을 부르지 않습니다. 친구마다 공개한 카드(`/<친구>/.well-known/agent-card.json`)를 읽어 이름·설명·스킬을 보여 줄 뿐입니다.
- 일정 확인과 약속 잡기는 이 노트북이 아니라 **서버의 친구 Agent**가 합니다. 그래서 도구 기록(`[messi] check_schedule()` 같은 줄)은 서버 터미널에 나오고, 노트북에는 내 Agent가 주고받은 메시지(`[나 → messi] …`, `[messi → 나] …`)가 보입니다.
- 내 Agent는 세 친구에게 한 번에 묻습니다(도구 호출 세 개를 함께 요청). 그래서 서버 터미널의 줄이 친구마다 섞여 나옵니다.
- 친구들의 말투와 문장은 실행할 때마다 조금씩 다릅니다. 정해진 일정에서 나온 요일·시간이 맞는지만 봅니다.
- 약속은 서버가 기억합니다. 약속을 잡은 뒤 같은 셀을 다시 실행하면 친구 일정에 `축구 약속(나)`이 남아 있어 결과가 처음과 달라집니다. 처음부터 다시 하려면 서버 터미널에서 Ctrl+C로 끄고 다시 띄운 뒤 실행합니다.

## 4. 정리

A2A로 내 Agent가 다른 Agent들에게 묻고 일을 맡겼습니다.

| 쪽 | 한 일 | 쓴 것 |
|---|---|---|
| 서버 (친구 Agent 세 명) | 요청을 받아 자기 일정 도구로 확인하고 약속을 잡아 답함 | AgentExecutor, Agent Card, RequestHandler, 친구마다 다른 주소 |
| 클라이언트 (내 Agent) | 카드를 읽고, 세 친구에게 묻고, 모두 비는 시간에 약속을 잡음 | A2ACardResolver, SendMessageRequest, `ask_friend` 도구 |

내 Agent는 친구들의 일정이 어떻게 저장되어 있는지 몰라도, 카드와 메시지 형식만 맞추면 묻고 맡길 수 있습니다. 친구 Agent 쪽을 바꾸면(2.2의 CHANGE_HERE, 호날두 말투) 내 Agent 코드를 고치지 않아도 답이 달라집니다.

**실습을 마치면 서버 터미널에서 Ctrl+C로 서버를 끕니다.** 같은 8001 포트를 쓰는 다른 서버(4일차 캡스톤 MCP 서버)를 띄울 때 부딪히지 않게 하기 위해서입니다.